# Spacing Statistics

## 1. Importing / Installing Packages

In [1]:
from __future__ import annotations

import os
from pathlib import Path

import pandas as pd
pd.set_option("display.max_columns", None) # Show all columns when printing DataFrames

import numpy as np
import math

import datetime

from matplotlib import pyplot as plt
# Ensures that plots are displayed inline in Jupyter notebooks
%matplotlib inline
%config InlineBackend.figure_format = 'svg' # Configuring inline backend to use SVG format for figures

from dataclasses import dataclass
from enum import Enum, auto

from typing import Dict, Tuple, List, Union, Optional, ClassVar, Any, Literal, Iterable

from src.utils import DatabricksOdbcConnector, reorder_columns, compute_bg_rcat, read_excel_with_mapper, standardize_column_names, read_csv_with_mapper
from src.well_data import WellDataLoader, GeoSurveyProcessor, DirectionalBenchNeighbors, WellSpacingCalculator, FloatingSectionWPS

## 2. Import Data

### 2.1 Reading Header Data From File

In [2]:
df_header_raw = read_excel_with_mapper(
    path=r"C:\Users\ApoorvaSaxena\OneDrive - Bandera Group\Desktop\Project AP\05. Spacing Study\01. EF - 74 Ranch\header_ranch_74_EF.xlsx",
    sheet_name="header",
    col_map={"API14": "uwi", "API12": "uwi12", "API10": "uwi10", "WellName":"well_name",
             "SpudDate":"spud_dt", "CompletionDate": "comp_dt",	"FirstProdDate":"first_prod_dt",
             "LastProdDate":"last_prod_dt", "ENVInterval":"bench"
             },
    dtype_map={"uwi": str, "uwi12": str, "uwi10": str, "well_name": str},
    parse_dates=["SpudDate", "CompletionDate", "FirstProdDate", "LastProdDate"] # Parsing date columns during import because it’s handled inside pd.read_excel before we rename.
)

### 2.2 Standardizing Columns

In [3]:
df_header_standardize = standardize_column_names(df_header_raw)

### 2.3 Reading Directional Survey From File

In [4]:
df_directional_survey_raw = read_csv_with_mapper(
    path=r"C:\Users\ApoorvaSaxena\OneDrive - Bandera Group\Desktop\Project AP\05. Spacing Study\01. EF - 74 Ranch\ds_wellbore_trajectories_ranch74_EF.CSV",
    col_map={"API Number":"uwi12", "Measured Depth":"md", "Inclination":"inclination", "Azimuth":"azimuth", 
             "True Vertical Depth":"tvd", "N_S":"N/S", "E_W":"E/W", "Latitude":"latitude", "Longitude":"longitude"},
    dtype_map={"uwi12": str},
    usecols=['API Number', 'Measured Depth', 'Inclination','Azimuth', 'True Vertical Depth', 'N_S', 'E_W', 'Latitude','Longitude']
)

### 2.4 Merge to add 'uwi' to directional survey data and reorder columns

In [5]:
df_directional_uwi = df_directional_survey_raw.merge(df_header_standardize[["uwi12","uwi"]], how='left', on='uwi12', copy=True).reset_index(drop=True)

df_directional_uwi = reorder_columns(df_directional_uwi, columns_to_move=["uwi"], reference_column="uwi12").copy()

## 3. Computing Reserve Category

### 3.1 Bg_Rscat

In [6]:
df_header_standardize["bg_rcat"] = compute_bg_rcat(df=df_header_standardize,
                col_map = {
        "status": "env_well_status",
        "last_prod": "last_prod_dt",
        "spud": "spud_dt",
        "comp": "comp_dt"
    })

### 3.2 Removing/Deleting rows from header data where rcat is blank

In [7]:
df_header_standardize = df_header_standardize[df_header_standardize["bg_rcat"] != ""].reset_index(drop=True).copy()

## 4. UTM Coordinates

### 4.1 Computing UTM Coordinates

In [8]:
# Initialize the GeoSurveyProcessor with the log directory
geo = GeoSurveyProcessor(log_dir=r"C:\Users\ApoorvaSaxena\OneDrive - Bandera Group\Desktop\Project AP\02.Python\well-spacing-analyzer\notebooks\logs")

[GeoLogger] INFO (12-08 06:05 PM): GeoSurveyProcessor initialized. (Line: 472) [well_data_manager.py]



In [9]:
df_utm = geo.compute_utm_coordinates(df=df_directional_uwi)

[GeoLogger] INFO (12-08 06:05 PM): ✅ Using lat/lon from input DataFrame. (Line: 648) [well_data_manager.py]

[GeoLogger] INFO (12-08 06:05 PM): ✅ UTM coordinate computation complete in 0.61 sec. (Line: 703) [well_data_manager.py]



### 4.2 Filtering to get only the lateral sections after the heel point

In [10]:
df_utm_lateral = geo.filter_after_heel_point(df=df_utm)

### 4.3 Calculate midpoints for lateral wells

In [11]:
df_midpoints_lateral = geo.get_heel_toe_midpoints_latlon(df=df_utm_lateral)

## 5. Spacing

### 5.1 Filtering header data to include certain RSV category and missing uwi in Directional Survey from header data

In [12]:
missing_uwi = set(df_header_standardize['uwi12']) - set(df_directional_uwi['uwi12'])
print(f"Number of missing UWI12: {len(missing_uwi)}")
# list(missing_uwi)

Number of missing UWI12: 226


In [13]:
df_header_filter_for_spacing = df_header_standardize[~(df_header_standardize["uwi12"].isin(missing_uwi)) & 
          (df_header_standardize["bg_rcat"].isin(["1PDP", "1PDSI", "1WOP"]))].reset_index(drop=True).copy()

### 5.2 Calculating Spacing

In [14]:
# Create an instance of WellSpacingCalculator with the filtered trajectories
spacing_stats_ik = WellSpacingCalculator(trajectories=df_utm_lateral[df_utm_lateral["uwi12"].isin(df_header_filter_for_spacing["uwi12"].unique())])

In [15]:
# spacing_stats_ik._calculate_spacing_statistics(
#     batch_size=200_000,
#     max_distance_miles=4.0,
#     save_batches_dir=r"C:\Users\ApoorvaSaxena\OneDrive - Bandera Group\Desktop\Project AP\05. Spacing Study\01. EF - 74 Ranch\SpacingStats",
#     max_crossline_ft=5_280
# )

### 5.3 Read Calculated and Saved Spacing Results from Parquet Files

In [16]:
df_spacing_ik = spacing_stats_ik._load_saved_batches(
    batch_folder=r"C:\Users\ApoorvaSaxena\OneDrive - Bandera Group\Desktop\Project AP\05. Spacing Study\01. EF - 74 Ranch\SpacingStats"
)

🔍 Found 1 batch files. Loading and combining...
✅ Loaded 131,308 rows from all batches.


In [17]:
df_spacing_ik

,well_i,well_k,horizontal_dist,horizontal_dist_median,vertical_dist,3D_dist,drill_direction_i,drill_direction_k,n_samples,dy_p5,angle_deg,pair_alignment,min_distance_ft,mean_windowed_ft,reject_reason,direction_axis,direction_to_k_from_i_axis,direction_axis_confidence,direction_axis_distribution,axis_forced,overlap_len_common_ft,LL_i,LL_k,overlap_pct_i,overlap_pct_k,proj_coverage_i_pct,contact_threshold_ft,contact_len_i_ft,contact_pct_i,contact_len_i_interior_ft,contact_pct_i_interior,contact_len_i_ft_T300,contact_pct_i_T300,contact_len_i_interior_ft_T300,contact_pct_i_interior_T300,horizontal_crossline_mean_ft,hz_effective,hz_basis,3D_dist_effective
0,42013355250000,42013355920000,NaN,NaN,NaN,NaN,NS,EW,NaN,NaN,12.998861,parallel_like,NaN,NaN,no_overlap_x,EW,None,NaN,,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,crossline_mean,NaN
1,42013355250000,42013355930000,NaN,NaN,NaN,NaN,NS,EW,NaN,NaN,9.280324,parallel_like,NaN,NaN,no_overlap_x,EW,None,NaN,,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,crossline_mean,NaN
2,42013355250000,42013355940000,NaN,NaN,NaN,NaN,NS,EW,NaN,NaN,5.675670,parallel_like,NaN,NaN,no_overlap_x,EW,None,NaN,,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,crossline_mean,NaN
3,42013355250000,42013355950000,NaN,NaN,NaN,NaN,NS,EW,NaN,NaN,2.120384,parallel_like,NaN,NaN,no_overlap_x,EW,None,NaN,,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,crossline_mean,NaN
4,42013355250000,42013355990000,NaN,NaN,NaN,NaN,NS,EW,NaN,NaN,4.744556,parallel_like,NaN,NaN,no_overlap_x,EW,None,NaN,,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,crossline_mean,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
131303,42311375330000,42311359010000,NaN,NaN,NaN,NaN,NS,EW,NaN,NaN,42.500148,oblique,NaN,NaN,coarse_far,EW,None,NaN,,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,min_nearest,NaN
131304,42311375330000,42311371500000,NaN,NaN,NaN,NaN,NS,EW,NaN,NaN,36.766096,oblique,NaN,NaN,coarse_far,EW,None,NaN,,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,min_nearest,NaN
131305,42311375330000,42311371510000,NaN,NaN,NaN,NaN,NS,EW,NaN,NaN,37.940408,oblique,NaN,NaN,coarse_far,EW,None,NaN,,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,min_nearest,NaN
131306,42311375330000,42311371520000,NaN,NaN,NaN,NaN,NS,EW,NaN,NaN,39.119606,oblique,NaN,NaN,coarse_far,EW,None,NaN,,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,min_nearest,NaN


### 5.4 Filtering out rows where 'reject_reason' is empty

In [18]:
# Filter out rows where 'reject_reason' is not empty
df_spacing_ik_filt = df_spacing_ik[df_spacing_ik["reject_reason"]==""].reset_index(drop=True).copy()

In [19]:
df_spacing_ik_filt

,well_i,well_k,horizontal_dist,horizontal_dist_median,vertical_dist,3D_dist,drill_direction_i,drill_direction_k,n_samples,dy_p5,angle_deg,pair_alignment,min_distance_ft,mean_windowed_ft,reject_reason,direction_axis,direction_to_k_from_i_axis,direction_axis_confidence,direction_axis_distribution,axis_forced,overlap_len_common_ft,LL_i,LL_k,overlap_pct_i,overlap_pct_k,proj_coverage_i_pct,contact_threshold_ft,contact_len_i_ft,contact_pct_i,contact_len_i_interior_ft,contact_pct_i_interior,contact_len_i_ft_T300,contact_pct_i_T300,contact_len_i_interior_ft_T300,contact_pct_i_interior_T300,horizontal_crossline_mean_ft,hz_effective,hz_basis,3D_dist_effective
0,42013355250000,42297351430000,3987.440677,6002.827212,188.935,3991.914276,NS,NS,57.0,NaN,25.177346,oblique,3987.440677,NaN,,EW,E,0.704997,"E:0.70,W:0.30",True,NaN,5617.295964,6384.722035,NaN,NaN,0.0,300.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,6056.616048,3987.440677,min_nearest,3991.914276
1,42013355250000,42297351500000,4034.932064,6037.948470,196.895,4039.733209,NS,NS,57.0,NaN,36.307540,oblique,4034.932064,NaN,,EW,E,0.687271,"E:0.69,W:0.31",True,NaN,5617.295964,6215.531717,NaN,NaN,0.0,300.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,6092.642338,4034.932064,min_nearest,4039.733209
2,42013355300000,42013355310000,476.291457,491.394274,14.640,476.516402,EW,NS,78.0,437.615761,1.583910,parallel_like,NaN,NaN,,NS,N,1.000000,"N:1.00,S:0.00",True,8236.524306,9326.804545,8271.672059,0.883102,0.995751,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,476.291457,476.291457,crossline_mean,476.516402
3,42013355300000,42013355320000,966.895082,994.570749,45.090,967.945870,EW,EW,72.0,838.163044,3.092182,parallel_like,NaN,NaN,,NS,N,1.000000,"N:1.00,S:0.00",True,7672.429810,9326.804545,7791.270805,0.822621,0.984747,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,966.895082,966.895082,crossline_mean,967.945870
4,42013355300000,42013355330000,1459.128905,1511.889831,39.675,1459.668205,EW,NS,63.0,1194.347148,5.133234,parallel_like,NaN,NaN,,NS,N,1.000000,"N:1.00,S:0.00",True,7134.439623,9326.804545,7350.734608,0.764939,0.970575,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1459.128905,1459.128905,crossline_mean,1459.668205
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
30297,42311375330000,42311369630000,3454.204506,3447.465844,172.720,3458.520054,NS,NS,65.0,3419.452352,2.587900,parallel_like,NaN,NaN,,EW,W,0.981640,"E:0.02,W:0.98",True,6527.187234,8865.176981,10814.012046,0.736273,0.603586,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3454.204506,3454.204506,crossline_mean,3458.520054
30298,42311375330000,42311371460000,81.768543,81.768543,330.230,340.202804,NS,NS,2.0,12.766229,9.713641,parallel_like,NaN,NaN,,EW,E,0.513875,"E:0.51,W:0.49",True,71.177818,8865.176981,11340.305646,0.008029,0.006277,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,81.768543,81.768543,crossline_mean,340.202804
30299,42311375330000,42311375300000,1489.314457,1496.436950,26.475,1489.549756,NS,EW,79.0,1446.933988,7.814606,parallel_like,NaN,NaN,,EW,E,1.000000,"E:1.00,W:0.00",True,7843.243603,8865.176981,10350.083425,0.884725,0.757795,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1489.314457,1489.314457,crossline_mean,1489.549756
30300,42311375330000,42311375310000,975.315525,972.658135,20.855,975.538469,NS,EW,80.0,940.544053,5.477167,parallel_like,NaN,NaN,,EW,E,1.000000,"E:1.00,W:0.00",True,7991.421917,8865.176981,9535.608570,0.901440,0.838061,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,975.315525,975.315525,crossline_mean,975.538469
